The Four Pillars

In [2]:
from google.genai import Client
from google.colab import userdata

# initialize the client
API_KEY = userdata.get('GEMINI_API_KEY')
client = Client(api_key=API_KEY)
model_id = "models/gemini-3-flash-preview"

# define the four pillars
instruction = "Classify the following customer support ticket into one of these categories: Billing, Technical, or General."
context = "You are a senior support coordinator. Be extremely concise. Do not provide an explanation."
input_data = "The mobile application crashes immediately after I open it. I have reinstalled the app, but the issue still persists."
output_indicator = "Category:"

# construct the full prompt with delimiters to help the model to distinguish the sections
full_prompt = f"""
### INSTRUCTION
{instruction}

### CONTEXT
{context}

### INPUT DATA
Ticket: {input_data}


### OUPUT
{output_indicator}
"""

# get the inference from the model
response = client.models.generate_content(
    model=model_id,
    contents=full_prompt
)

print(f"Final Output: {response.text}")

Final Output: Technical


Zero Shot Prompting

In [3]:
from google.genai import Client
from google.colab import userdata

# initialize the client
API_KEY = userdata.get('GEMINI_API_KEY')
client = Client(api_key=API_KEY)
model_id = "models/gemini-3-flash-preview"

# zero shot prompting: no examples provided, just raw instructions.
zero_shot_prompt = """
Analyze the following customer message.
Return the 'Sentiment' (Positive/Negative/Neutral) and 'Urgency' (High/Medium/Low).

Message: "I've been waiting for three days for a response to my ticket. This is ridiculous and I need this fixed by end of day or I'm canceling my service."

Output:
"""

# get the inference from the model
response = client.models.generate_content(
    model=model_id,
    contents=zero_shot_prompt
)

print(f"Zero-Shot Result: \n{response.text}")

Zero-Shot Result: 
Sentiment: Negative
Urgency: High


Few Shot Prompting

In [6]:
from google.genai import Client
from google.colab import userdata

# initialize the client
API_KEY = userdata.get('GEMINI_API_KEY')
client = Client(api_key=API_KEY)
model_id = "models/gemini-3-flash-preview"

# zero shot prompting: no examples provided, just raw instructions.
few_shot_prompt = """
Classify customer feedback by Category and Sentiment.
Follow the exact pattern shown in the examples.

Example 1:
Input: "The app keeps crashing whenever I try to upload a photo."
Output: CATEGORY: Technical | SENTIMENT: Negative

Example 2:
Input: "I love the new dark mode feature, it looks amazing!"
Output: CATEGORY: Product | SENTIMENT: Positive

Example 3:
Input: "Why was I charged $20 extra this month? I want an explanation."
Output: CATEGORY: Billing | SENTIMENT: Negative

Input: "The delivery person was very rude and left the package in the rain."
Output:
"""

# get the inference from the model
response = client.models.generate_content(
    model=model_id,
    contents=few_shot_prompt
)

print(f"Few-Shot Result: \n{response.text}")

Few-Shot Result: 
CATEGORY: Logistics | SENTIMENT: Negative


In [7]:
from google.genai import Client
from google.genai import types # Import types for configuration
from google.colab import userdata

# Initialize Client
API_KEY = userdata.get('GEMINI_API_KEY')
client = Client(api_key=API_KEY)
model_id = "models/gemini-3-flash-preview"

# 1. THE SYSTEM PERSONA (The Role)
# This stays constant and 'steers' the model throughout the interaction.
system_persona = """
You are a Senior Technical Support Engineer at a Tier-1 Cloud Provider.
Your tone is professional, calm, and highly technical.
You prioritize security and efficiency.
When asked a question, you follow a 'Triage' approach:
1. Identify the likely cause.
2. Provide a 3-step immediate mitigation.
3. Advise on long-term prevention.
"""

# 2. THE USER PROMPT (The Task)
user_query = "Our database is showing 100% CPU usage and customers can't log in. What do I do?"

# 3. THE INFERENCE
# We pass the persona in the 'config' to keep it separate from the conversation.
response = client.models.generate_content(
   model=model_id,
   contents=user_query,
   config=types.GenerateContentConfig(
       system_instruction=system_persona
   )
)

print(f"Senior Engineer Response:\n{'-'*30}\n{response.text}")

Senior Engineer Response:
------------------------------
This is a Critical Severity (Sev-1) event. Based on your description, the database is likely experiencing **Resource Exhaustion** driven by either an unoptimized query executing at high frequency, a sudden spike in connection attempts (connection storm), or a missing index on a high-traffic table.

### 1. Likely Cause Identification
The most common causes for 100% CPU on a production database are:
*   **Inefficient Execution Plans:** A recent code deployment or data volume change has caused a frequent query to perform a full table scan instead of an index seek.
*   **Connection Exhaustion:** A massive influx of login attempts is overwhelming the database’s process scheduler, often caused by application-side connection pooling failures.
*   **Contention/Locking:** High-row contention causing the CPU to spend cycles managing transaction locks and deadlocks rather than processing data.

---

### 2. Immediate Mitigation (3-Step Triag